# SmartHire — 02. Supervised Resume Category Classifier
In this notebook, we implement, benchmark, and evaluate:
- **TF-IDF Feature Representation** (n-grams: 1-2, sublinear term frequency, English stopword removal).
- **Model 1: Logistic Regression** (multinomial / OvR with L2 regularization).
- **Model 2: Linear SVM** (calibrated Support Vector Classifier).
- **Stratified 80/20 Train/Test Split** ensuring balanced class representation across all 25 categories.
- **Full Empirical Metrics**: Accuracy, Macro/Weighted Precision, Recall, F1-score, and Confusion Matrix.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import CLASSIFIER_PATH, CLASSIFIER_VEC_PATH, RANDOM_STATE, RESUMES_PROCESSED_PATH, TEST_SIZE
from src.evaluate import evaluate_classifier
from src.models.classifier import ResumeClassifier

df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "resumes_clean.csv")
print(f"Cleaned resumes loaded: {df.shape}")
print(f"Total categories: {df['category'].nunique()}")


## 1. Stratified Train/Test Split


In [ ]:
X_raw = df['cleaned_text'].tolist()
y = df['category'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)
print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")


## 2. Train and Evaluate Logistic Regression


In [ ]:
clf_lr = ResumeClassifier()
clf_lr.fit(X_train, y_train, model_type="logistic_regression")

X_test_vec_lr = clf_lr.vectorizer.transform(X_test)
y_pred_lr = clf_lr.model.predict(X_test_vec_lr)

classes = sorted(list(set(y)))
metrics_lr = evaluate_classifier(np.array(y_test), y_pred_lr, class_names=classes, save_cm_plot=True)

print(f"Logistic Regression Accuracy: {metrics_lr['accuracy'] * 100:.2f}%")
print(f"Weighted Precision: {metrics_lr['precision_weighted'] * 100:.2f}%")
print(f"Weighted Recall:    {metrics_lr['recall_weighted'] * 100:.2f}%")
print(f"Weighted F1-Score:  {metrics_lr['f1_weighted'] * 100:.2f}%")


## 3. Train and Evaluate Linear SVM


In [ ]:
clf_svm = ResumeClassifier()
clf_svm.fit(X_train, y_train, model_type="linear_svm")

X_test_vec_svm = clf_svm.vectorizer.transform(X_test)
y_pred_svm = clf_svm.model.predict(X_test_vec_svm)

metrics_svm = evaluate_classifier(np.array(y_test), y_pred_svm, class_names=classes, save_cm_plot=False)

print(f"Linear SVM Accuracy: {metrics_svm['accuracy'] * 100:.2f}%")
print(f"Weighted Precision: {metrics_svm['precision_weighted'] * 100:.2f}%")
print(f"Weighted Recall:    {metrics_svm['recall_weighted'] * 100:.2f}%")
print(f"Weighted F1-Score:  {metrics_svm['f1_weighted'] * 100:.2f}%")


## 4. Model Comparison Table


In [ ]:
comparison_df = pd.DataFrame([
    {
        "Algorithm": "Logistic Regression (Primary)",
        "Accuracy": f"{metrics_lr['accuracy'] * 100:.2f}%",
        "Precision (Weighted)": f"{metrics_lr['precision_weighted'] * 100:.2f}%",
        "Recall (Weighted)": f"{metrics_lr['recall_weighted'] * 100:.2f}%",
        "F1 (Weighted)": f"{metrics_lr['f1_weighted'] * 100:.2f}%",
    },
    {
        "Algorithm": "Linear SVM (Calibrated)",
        "Accuracy": f"{metrics_svm['accuracy'] * 100:.2f}%",
        "Precision (Weighted)": f"{metrics_svm['precision_weighted'] * 100:.2f}%",
        "Recall (Weighted)": f"{metrics_svm['recall_weighted'] * 100:.2f}%",
        "F1 (Weighted)": f"{metrics_svm['f1_weighted'] * 100:.2f}%",
    }
])
display(comparison_df)


## 5. Live Inference on a Sample Resume


In [ ]:
sample_resume = '''
Machine Learning Engineer with 3 years experience.
Proficient in Python, PyTorch, Scikit-learn, Pandas, NumPy, Deep Learning, SQL, and Git.
Built convolutional neural networks and transformer NLP pipelines.
'''
prediction = clf_lr.predict(sample_resume)
print("Predicted Category:", prediction['predicted_category'])
print("Confidence:", f"{prediction['confidence'] * 100:.1f}%")
print("Top 3 Predicted Domains:", prediction['top_categories'])
print("Explanation:", prediction['explanation'])
